# Pressure containment


## Graphtik

### design pressure
e



###
on Wind NOTE
UnicodeEncodeError: 'charmap' codec can't encode character '\u03b1' in position 3695: character maps to <und
UnicodeEncodeError: 'charmap' codec can't encode character '\u03c1' in position 3196: character maps to <undefined>  efined>
https://stackoverflow.com/questions/50933194/how-do-i-set-the-pythonutf8-environment-variable-to-enable-utf-8-encodin  g-by-def
Graphtik plot: To plot graphs on Windows, set the following enviromental varibale  before launching Jupyter no  tebook   
set PYTHONUTF8=1  
set YHONUTF8="1"   avoid trailing space 

In [55]:
from graphtik import compose, operation, keyword
from pdover2t.dnvstf101 import *
from pdover2t.pipe import dowt2di

use_numpy = True
if use_numpy:
    import numpy as np

import pandas as pd

# set PYTHONUTF8=1
create_graph_images = False

In [56]:
params = {
    "D_o": 24 * 25.4 * 1.e-3,  # (m) pipe internal diameter
    "t_nom": 0.0159,  # (m) pipe wall thickness
    "t_corr": 0.0, # (m) corrosion allowance
    "t_fab": 0.001, # (m) thickness negative fabrication tolerance

    "SMYS": 450.e6, # (Pa) pipe steel SMYS
    "f_ytemp": 35.e6, # (Pa) steel yield strength temperature de-rating
    "SMTS": 535.e6, # (Pa) pipe steel SMTS
    "f_utemp": 0.e6, # (Pa) steel ultimate strength temperature de-rating
    "α_U": 0.96, # material strength factor
    "α_U_pt": 1.0, # material strength factor for prssure test cacls
    "γ_m": 1.15,  # material resistance factor

    "p_d": 150.e5,  # (Pa) design pressure at reference elevation Z_ref
    "γ_inc": 1.10 , # incidental to design pressure ratio
    "h_ref": 0.0 , # (m) reference elevation for pressure (LAT=0m)
    "ρ_cont": 20., # (kg/m3) density of pipeline contents
    "ρ_t": 1025. ,  # test fluid density

    "ρ_seawater": 1025., # (kg/m3) density of seawater

    "α_spt": 1.05,  # DNVGL-ST-F101 (2017-12) p94gamma
    "α_mpt": 1.038,  # p94
    "α_spt": 1.05,  # p94
    "γ_SCPC": 1.138,  # safety class resistance factor for pressure containment

    "mill_test_k": 1.0,   # k parameter used for calculating mill test pressure
    "t_corr_mill_test": 0.0,

    "MSL": 55.0,
    "tide": 1.0,
}

In [57]:
if use_numpy:
    params["p_d"] = np.array([50.e5, 150.e5, 250.e5])

In [58]:
cgraph = compose("start",
  operation(lambda sea_level,tide: -(sea_level - tide), name="seabed level", needs=["MSL", "tide"], provides=["h_l"]),
  operation(char_WT, name="characteristic WT", needs=["t_nom", "t_fab", "t_corr"],  provides=["t_1"]),
    operation(p_ext, name="external pressure", needs=["h_l", "ρ_seawater"],  provides=["p_e"])
)

In [59]:
cgraph= compose("material strength", cgraph,
    operation(char_strength, name="char_fy", needs=["SMYS", "α_U", keyword("f_ytemp")], provides=["f_y"]), 
    operation(char_strength, name="char_fu", needs=["SMTS", "α_U", keyword("f_utemp", "f_ytemp")], provides=["f_u"])
)

In [60]:
cgraph = compose("pressure test", cgraph,
    operation(p_incid_loc, name="local_incidental_pressure", needs=["p_d", "ρ_cont", "h_l", "h_ref", "γ_inc"], provides=["p_li"]),
    operation(p_system_test_ref, name="test pressure", needs=["p_d", "γ_inc", "α_spt"], provides=["p_t"]),
    operation(p_test_loc, name="local test pressure", needs=["p_t", "ρ_t", "h_l", "h_ref"], provides=["p_lt"])
)

In [61]:
cgraph= compose("incidental pressure", cgraph, 
   # operation(p_incid_ref, name="incidental pressure", needs=["p_d", "γ_inc"], provides=["p_inc"]),
    operation(p_incid_loc, name="local_incidental_pressure", needs=["p_d", "ρ_cont", "h_l", "h_ref", "γ_inc"], provides=["p_li"]),
    operation(p_test_loc_uty2, name="pli hydrotest", needs=["α_spt", "p_lt", "p_li"], provides=["pli_lt_uty"])
)

In [62]:
cgraph= compose("mill test calcs", cgraph, 
   operation(char_WT, name="t_min for mill_test", needs=["t_nom", "t_fab", "t_corr_mill_test"],  provides=["t_min_mill_test"]),
   operation(p_mill_test, name="mill_test pressure", needs=["D_o", "t_min_mill_test", "SMYS", "SMTS", "α_U", "α_mpt", keyword("mill_test_k", "k")],  provides=["p_mpt"]),
   operation(p_mill_test_uty, name="pli mill test unity", needs=["p_li", "p_e", "p_mpt"], provides=["pli_mpt_uty"])
)

In [63]:
cgraph= compose("pt material strength", cgraph,
    operation(char_strength, name="pt_char_fy", needs=["SMYS", "α_U_pt", keyword("f_ytemp")], provides=["pt_f_y"]), 
    operation(char_strength, name="pt_char_fu", needs=["SMTS", "α_U_pt", keyword("f_utemp", "f_ytemp")], provides=["pt_f_u"])
)

In [64]:
cgraph= compose("pressure containment (operation)", cgraph, 
  operation(p_contain_resist, name="pt p_b", needs=["D_o", "t_1", "pt_f_y", "pt_f_u"], provides=["pt_p_b"]),
  operation(p_mill_test, name="pt mill_test pressure", needs=["D_o", "t_min_mill_test", "SMYS", "SMTS", "α_U_pt", "α_mpt", keyword("mill_test_k", "k")],  provides=["pt_p_mpt"]),
  operation(p_mill_test_uty, name="plt mill test unity", needs=["p_lt", "p_e", "pt_p_mpt"], provides=["plt_mpt_uty"]),
  operation(p_contain_resist_uty, name="pt containment resistance unity", needs=["p_lt", "p_e",  "pt_p_b", "γ_m", "γ_SCPC"],  provides=["plt_cont_res_uty"]),
  operation(p_contain_uty2, name="pt containment resistance", needs=["plt_cont_res_uty", "plt_mpt_uty"], provides=["plt_cont_uty"])
)

In [65]:
cgraph= compose("pressure containment (operation)", cgraph, 
  operation(p_contain_uty2, name="press contain unity check", needs=["pli_cont_uty", "plt_cont_uty"], provides=["press_contain_unity"])
)

In [66]:
if create_graph_images:
    cgraph.plot("design-pressure_test1.svg")

In [67]:
result = cgraph.compute(params)
if create_graph_images:
    result.plot("design-pressure_test2.svg")

In [68]:
try:
    df = pd.DataFrame(data=dict(result))
except ValueError:
    df = pd.DataFrame(data=dict(result), index=[0]) # ValueError: If using all scalar values, you must pass an index
df

,D_o,t_nom,t_corr,t_fab,SMYS,f_ytemp,SMTS,f_utemp,α_U,α_U_pt,...,t_min_mill_test,p_mpt,pli_mpt_uty,pt_f_y,pt_f_u,pt_p_b,pt_p_mpt,plt_mpt_uty,plt_cont_res_uty,plt_cont_uty
0,0.6096,0.0159,0.0,0.001,450000000.0,35000000.0,535000000.0,0.0,0.96,1.0,...,0.0149,2.002055e+07,0.248135,415000000.0,535000000.0,2.401241e+07,2.085474e+07,0.276916,0.314743,0.314743
1,0.6096,0.0159,0.0,0.001,450000000.0,35000000.0,535000000.0,0.0,0.96,1.0,...,0.0149,2.002055e+07,0.797570,415000000.0,535000000.0,2.401241e+07,2.085474e+07,0.830747,0.944229,0.944229
2,0.6096,0.0159,0.0,0.001,450000000.0,35000000.0,535000000.0,0.0,0.96,1.0,...,0.0149,2.002055e+07,1.347006,415000000.0,535000000.0,2.401241e+07,2.085474e+07,1.384578,1.573716,1.573716
